In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/diabetes-health-indicators-dataset/diabetes_dataset.csv
/kaggle/input/playground-series-s5e12/sample_submission.csv
/kaggle/input/playground-series-s5e12/train.csv
/kaggle/input/playground-series-s5e12/test.csv


In [2]:
class CONFIG:
    INPUT_DIR = '/kaggle/input/playground-series-s5e12'
    
    N_FOLDS = 5
    SEED = 42

    TARGET = 'diagnosed_diabetes'

config = CONFIG()

train = pd.read_csv(f'{config.INPUT_DIR}/train.csv')
test = pd.read_csv(f'{config.INPUT_DIR}/test.csv')
# test[config.TARGET] = -1
train['source'] = 'train'
test['source']= 'test'

combine = pd.concat([train, test])
submission = pd.read_csv(f'{config.INPUT_DIR}/sample_submission.csv')

In [3]:
def eda(df, name):
    print(f"Exploring {name} dataframe")
    print('='*30)
    print(f'\n NULL VALUES: {df.isnull().sum()}')
    print('='*30)
    print(f'\n {name} dataframe shape: {df.shape}')
    print('='*30)
    print(f'\n {name} dataframe numerical stats: {df.describe()}')
    print('='*30)

eda(train, 'TRAIN')
eda(test, 'TEST')

Exploring TRAIN dataframe

 NULL VALUES: id                                    0
age                                   0
alcohol_consumption_per_week          0
physical_activity_minutes_per_week    0
diet_score                            0
sleep_hours_per_day                   0
screen_time_hours_per_day             0
bmi                                   0
waist_to_hip_ratio                    0
systolic_bp                           0
diastolic_bp                          0
heart_rate                            0
cholesterol_total                     0
hdl_cholesterol                       0
ldl_cholesterol                       0
triglycerides                         0
gender                                0
ethnicity                             0
education_level                       0
income_level                          0
smoking_status                        0
employment_status                     0
family_history_diabetes               0
hypertension_history                  0

In [4]:
FEATURES = [col for col in train.columns if col not in ['id', 'diagnosed_diabetes']]
CATS = train[FEATURES].select_dtypes(include='object').columns.to_list()
NUMS = train[FEATURES].select_dtypes(include=['int64', 'float64']).columns.to_list()

combine = pd.concat([train, test])

In [5]:
CATS1 = []

for c in NUMS:
    n = f'{c}_cat'
    for df in [combine]:
        df[n] = df[c].astype('category')
        
    CATS1.append(n)

print(CATS1)

['age_cat', 'alcohol_consumption_per_week_cat', 'physical_activity_minutes_per_week_cat', 'diet_score_cat', 'sleep_hours_per_day_cat', 'screen_time_hours_per_day_cat', 'bmi_cat', 'waist_to_hip_ratio_cat', 'systolic_bp_cat', 'diastolic_bp_cat', 'heart_rate_cat', 'cholesterol_total_cat', 'hdl_cholesterol_cat', 'ldl_cholesterol_cat', 'triglycerides_cat', 'family_history_diabetes_cat', 'hypertension_history_cat', 'cardiovascular_history_cat']


In [6]:
CATS2 = []
SIZES = {}

for c in CATS+CATS1:
    n = f'{c}_enc'
    for df in [combine]:
        df[c] = df[c].astype('category')
        df[n], _ =  df[c].factorize()
        df[n] = df[n].astype('float32')
        s = df[n].max()+1

    CATS2.append(n)
    SIZES[n] = s

print(CATS2)
print('='*30)
print(f'CARDINALITY OF CATS2: {SIZES}')

['gender_enc', 'ethnicity_enc', 'education_level_enc', 'income_level_enc', 'smoking_status_enc', 'employment_status_enc', 'source_enc', 'age_cat_enc', 'alcohol_consumption_per_week_cat_enc', 'physical_activity_minutes_per_week_cat_enc', 'diet_score_cat_enc', 'sleep_hours_per_day_cat_enc', 'screen_time_hours_per_day_cat_enc', 'bmi_cat_enc', 'waist_to_hip_ratio_cat_enc', 'systolic_bp_cat_enc', 'diastolic_bp_cat_enc', 'heart_rate_cat_enc', 'cholesterol_total_cat_enc', 'hdl_cholesterol_cat_enc', 'ldl_cholesterol_cat_enc', 'triglycerides_cat_enc', 'family_history_diabetes_cat_enc', 'hypertension_history_cat_enc', 'cardiovascular_history_cat_enc']
CARDINALITY OF CATS2: {'gender_enc': 3.0, 'ethnicity_enc': 5.0, 'education_level_enc': 4.0, 'income_level_enc': 5.0, 'smoking_status_enc': 3.0, 'employment_status_enc': 4.0, 'source_enc': 2.0, 'age_cat_enc': 71.0, 'alcohol_consumption_per_week_cat_enc': 9.0, 'physical_activity_minutes_per_week_cat_enc': 578.0, 'diet_score_cat_enc': 99.0, 'sleep_hou

In [7]:
from itertools import combinations

INTER = []

for col1, col2 in combinations(CATS+CATS1, 2):
    n = f'{col1}_{col2}_inter'

    for df in [combine]:
        df[n] = df[col1].astype(str) + '_' + df[col2].astype(str)
        df[n] = df[n].astype('category')

    INTER.append(n)

# print(INTER)
print('='*30)
print('LENGTH OF INTER :', len(INTER))

LENGTH OF INTER : 300


In [8]:
train_idx = combine['source'] == 'train'
test_idx = combine['source'] == 'test'

train_n = combine[train_idx].reset_index(drop=True).copy()
test_n = combine[test_idx].reset_index(drop=True).copy()

In [9]:
test_n[CATS]

,gender,ethnicity,education_level,income_level,smoking_status,employment_status,source
0,Female,White,Highschool,Middle,Former,Employed,test
1,Female,White,Highschool,Middle,Never,Unemployed,test
2,Male,White,Highschool,Low,Never,Employed,test
3,Male,White,Graduate,Middle,Former,Employed,test
4,Male,White,Graduate,Low,Current,Unemployed,test
...,...,...,...,...,...,...,...
299995,Male,White,Highschool,Upper-Middle,Former,Employed,test
299996,Male,Asian,Postgraduate,Lower-Middle,Never,Employed,test
299997,Female,White,Highschool,Middle,Never,Employed,test
299998,Male,White,Highschool,Low,Current,Retired,test


In [10]:
FEATURES_T = CATS + CATS1 + CATS2 + NUMS + INTER
len(FEATURES_T)

368

In [11]:
from sklearn.base import BaseEstimator, TransformerMixin

class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols_to_encode, aggs=['mean'], cv=5, smooth='auto', drop_original=True):
        self.cols_to_encode = cols_to_encode
        self.aggs = aggs
        self.cv = cv
        self.smooth = smooth
        self.drop_original = drop_original
        self.mappings_ = {}
        self.global_stats_ = {}

    def fit(self, X, y):
        temp_df = X.copy()
        temp_df['target'] = y

        for agg_func in self.aggs:
            self.global_stats_[agg_func] = y.agg(agg_func)

        for col in self.cols_to_encode:
            self.mappings_[col] = {}
            for agg_func in self.aggs:
                mapping = temp_df.groupby(col)['target'].agg(agg_func)
                self.mappings_[col][agg_func] = mapping

        return self

    def transform(self, X):
        X_transformed = X.copy()
        for col in self.cols_to_encode:
            for agg_func in self.aggs:
                new_col_name = f'TE_{col}_{agg_func}'
                map_series = self.mappings_[col][agg_func]
    
                # 1. map, 2. cast to float, 3. fill NaN
                X_transformed[new_col_name] = (
                    X[col].map(map_series)
                         .astype(float)          # <-- key line
                         .fillna(self.global_stats_[agg_func])
                )
    
        if self.drop_original:                   # typo fixed
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)
    
        return X_transformed

    def fit_transform(self, X, y):

        self.fit(X, y)
        encoded_features = pd.DataFrame(index=X.index)

        skf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=42)

        for train_idx, val_idx in skf.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[val_idx]
            X_val = X.iloc[val_idx]

            temp_df_train = X_train.copy()
            temp_df_train['target'] = y_train

            for col in self.cols_to_encode:
                for agg_func in self.aggs:
                    new_col_name = f'TE_{col}_{agg_func}'

                    fold_global_stats = y_train.agg(agg_func)
                    mapping = temp_df_train.groupby(col)['target'].agg(agg_func)

                    if agg_func == 'mean':
                        counts = temp_df_train.groupby(col)['target'].count()

                        m = self.smooth
                        if self.smooth == 'auto':
                            variance_between = mapping .var()
                            avg_variance_within = temp_df_train.groupby(col)['target'].mean()
                            if variance_between > 0:
                                m = avg_variance_within / variance_between

                            else:
                                m = 0

                        smoothed_mapping = (counts * mapping + m * fold_global_stats) / (counts + m)

                        encoded_values = X_val[col].map(smoothed_mapping)

                    else:
                        encoded_values = X_val[col].map(mapping)

                    encoded_features.loc[X_val.index, new_col_name] = encoded_values.fillna(fold_global_stats)

            X_transformed = X.copy()
            for col in encoded_features.columns:
                X_transformed[col] = encoded_features[col]

            if self.drop_original:
                X_transformed.drop(columns=self.cols_to_encode, inplace=True)

            return X_transformed
                    
                    

In [12]:
X = train_n[FEATURES_T].copy()
y = train_n[config.TARGET]

test_n = test_n[FEATURES_T].copy()

In [13]:
print('gender_ethnicity_inter' in test_n.columns) 

True


In [14]:
from xgboost import XGBClassifier
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 5,
    'colsample_bytree': 0.5,
    'subsample': 0.8,
    'n_estimators': 10000,
    'learning_rate': 0.01,
    'early_stopping_rounds': 100,
    'random_state': 42,
    'n_jobs': -1,
    'device': 'cuda',
    'enable_categorical': True,
}

skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    print(f'TRAINING SHAPE BEFORE ENCODING :{X_train.shape}')
    TE = TargetEncoder(cols_to_encode=INTER, aggs=['mean', 'count'], cv=5, smooth='auto', drop_original=True)
    X_train = TE.fit_transform(X_train, y_train)
    X_val = TE.transform(X_val)
    test_enc = TE.transform(test_n)
    print(f'TRAINING SHAPE AFTER ENCODING :{X_train.shape}')
    model = XGBClassifier(**params)

    model.fit(X_train, y_train,
             eval_set=[(X_val, y_val)],
             verbose=1000)

    val_preds = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = val_preds

    fold_score = roc_auc_score(y_val, val_preds)
    print(f'FOLD {fold} AUC: {fold_score:.4f}')
    test_preds +=  model.predict_proba(test_enc)[:, 1] / config.N_FOLDS

overall_auc = roc_auc_score(y, oof_preds)
print('='*30)
print(f"Overall OOF AUC: {overall_auc:.4f}")
print('='*30)

TRAINING SHAPE BEFORE ENCODING :(560000, 368)
TRAINING SHAPE AFTER ENCODING :(560000, 668)
[0]	validation_0-auc:0.68026
[1000]	validation_0-auc:0.73058
[1421]	validation_0-auc:0.73069
FOLD 0 AUC: 0.7307
TRAINING SHAPE BEFORE ENCODING :(560000, 368)
TRAINING SHAPE AFTER ENCODING :(560000, 668)
[0]	validation_0-auc:0.67758
[1000]	validation_0-auc:0.72944
[1881]	validation_0-auc:0.72969
FOLD 1 AUC: 0.7297
TRAINING SHAPE BEFORE ENCODING :(560000, 368)
TRAINING SHAPE AFTER ENCODING :(560000, 668)
[0]	validation_0-auc:0.67883
[1000]	validation_0-auc:0.73003
[1829]	validation_0-auc:0.73040
FOLD 2 AUC: 0.7304
TRAINING SHAPE BEFORE ENCODING :(560000, 368)
TRAINING SHAPE AFTER ENCODING :(560000, 668)
[0]	validation_0-auc:0.67830
[1000]	validation_0-auc:0.73023
[1876]	validation_0-auc:0.73055
FOLD 3 AUC: 0.7306
TRAINING SHAPE BEFORE ENCODING :(560000, 368)
TRAINING SHAPE AFTER ENCODING :(560000, 668)
[0]	validation_0-auc:0.67940
[1000]	validation_0-auc:0.72976
[1501]	validation_0-auc:0.73001
FOLD

In [15]:
test_n.shape

(300000, 368)

In [16]:
submission[config.TARGET] = test_preds
submission.to_csv(f'submission_cv_{overall_auc}.csv', index=False)